# F2 · Preprocesamiento de licitaciones de ChileCompra

**Pregunta descriptiva propuesta.** ¿Cómo se distribuye la diferencia entre el monto estimado registrado y la suma de las ofertas ganadoras por licitación, por moneda y visibilidad del estimado? ¿En qué proporción coinciden ambos montos?

La brecha describe diferencias entre valores registrados. Para interpretarla como desviación de un presupuesto previo haría falta verificar que el estimado se fijó antes de conocer la adjudicación. La base de ofertas se mantiene a nivel de oferta por ítem. La brecha se guarda en una tabla aparte, con una fila por licitación.

## 1. Configurar el proyecto

Descarga en [Datos Abiertos de ChileCompra](https://datos-abiertos.chilecompra.cl/descargas) el CSV de licitaciones correspondiente al período del proyecto. Guárdalo localmente como `data/raw/04LicitacionesEne.csv`. Mantén ese directorio fuera de Git y conserva el año y mes exactos en el README para que otra persona pueda repetir la descarga.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

actual = Path.cwd()
raiz = next((
    carpeta for carpeta in (actual, *actual.parents)
    if (carpeta / "src/funciones_f2_revision.py").is_file()
), None)
if raiz is None:
    raise FileNotFoundError("Ubica funciones_f2_revision.py y pruebas_f2.py en src/ del proyecto.")
sys.path.insert(0, str(raiz))
from src.funciones_f2_revision import (
    MOTIVOS, aplicar_bandera_atipicos, calcular_brecha, calcular_limites_iqr,
    cargar_csv, convertir_tipos, marcar_duplicados, normalizar_codigo,
    ordenar_tamano, perfilar_columnas, tratar_faltantes, validar_procesado,
)
from src.pruebas_f2 import ejecutar_pruebas_f2

ruta = raiz / "data/raw/04LicitacionesEne.csv"
print("Entrada:", ruta.relative_to(raiz))
print("Versión de pandas:", pd.__version__)

## 2. Verificar las reglas con datos ficticios

In [ ]:
ejecutar_pruebas_f2()
print("Pruebas de brecha, duplicados, tamaño, esquema e IQR aprobadas.")

## 3. Cargar, perfilar y revisar duplicados

Se excluyen los identificadores directos detectados en los encabezados al leer. Se marcan las coincidencias exactas de las columnas restantes **antes de transformar tipos o recortar columnas**. Tras excluir identificadores, dos filas pueden verse iguales y corresponder a ofertas legítimas. Se conservan las filas y se reporta cuántas coincidencias involucran ofertas ganadoras. Se compara después la brecha con y sin licitaciones afectadas para medir sensibilidad sin eliminar ofertas automáticamente.

In [ ]:
datos = cargar_csv(ruta)
filas_originales = len(datos)
print(f"Entrada: {filas_originales:,} filas y {datos.shape[1]} columnas")
perfil = perfilar_columnas(datos)
display(perfil)
nulos_entrada = datos.isna().sum()
datos, resumen_duplicados = marcar_duplicados(datos)
print("Filas adicionales idénticas por resultado:")
display(resumen_duplicados)

## 4. Convertir y decidir qué hacer con los faltantes

Se separan las ausencias originales de los fallos de conversión. Para cada variable del análisis se presenta su regla y motivo. Las columnas con más del 90 % de faltantes se revisan por separado. La exportación de ofertas usa una lista explícita de variables analíticas y excluye campos de texto libre como nombres de ofertas y descripciones.

In [ ]:
datos, conversiones = convertir_tipos(datos)
display(conversiones)
datos, faltantes = tratar_faltantes(datos, nulos_entrada.reindex(datos.columns))
faltantes = faltantes.drop(index="EsDuplicadoPotencial")
display(faltantes.loc[faltantes.index.intersection(MOTIVOS)])
display(faltantes.loc[faltantes["% faltantes"].gt(90)].sort_values("% faltantes", ascending=False))
datos = ordenar_tamano(datos)
display(datos["TamanoProveedorOrdinal"].value_counts(dropna=False).sort_index())

## 5. Calcular la brecha por licitación

Se suman los montos totales de **todas las ofertas ganadoras** por licitación. El estimado debe ser positivo, estar presente y ser consistente en las filas de la licitación. Se requiere una moneda de licitación consistente y que cada ganadora tenga la misma moneda y un monto positivo. También se excluyen licitaciones con resultados de oferta faltantes para evitar sumas parciales.

**Brecha = suma de montos ganadores − monto estimado registrado.** Se reporta la cobertura por licitación y se resume por moneda. Para identificar coincidencias se usa una tolerancia absoluta de 0,00000001 por el redondeo numérico. La comparación de sensibilidad excluye licitaciones con coincidencias exactas entre filas ganadoras.

In [ ]:
licitaciones = calcular_brecha(datos)
comparables = licitaciones.loc[licitaciones["EstadoBrecha"].eq("Calculada")]
sin_duplicadas = comparables.loc[~comparables["GanadoraDuplicada"]]

print(f"Licitaciones con código: {len(licitaciones):,}")
print(f"Filas sin código: {normalizar_codigo(datos['NroLicitacion']).isna().sum():,}")
display(licitaciones["EstadoBrecha"].value_counts().to_frame("Licitaciones"))
print(f"Brechas calculadas: {len(comparables):,} de {len(licitaciones):,}")
if len(licitaciones):
    print(f"Cobertura: {len(comparables) / len(licitaciones):.1%}")
for nombre, muestra in [("Todas las comparables", comparables), ("Sin ganadoras duplicadas potenciales", sin_duplicadas)]:
    print(nombre)
    display(muestra.groupby("MonedaLicitacion").agg(
        Licitaciones=("BrechaMonto", "size"),
        Coinciden=("CoincideConEstimado", "sum"),
        ProporcionCoincide=("CoincideConEstimado", "mean"),
        MedianaBrecha=("BrechaMonto", "median"),
    ))

## 6. Revisar si el estimado sirve como referencia previa

El [diccionario de licitaciones de la API de Mercado Público](https://www.chilecompra.cl/wp-content/uploads/2026/03/Documentacion-API-Mercado-Publico-Licitaciones.pdf) describe el monto estimado de la institución y un indicador que determina si se publica en la ficha. Es documentación de la API. No establece cuándo se registró el valor en este extracto CSV ni confirma su independencia de lo adjudicado.

Se contrasta la coincidencia por moneda y visibilidad. Los campos `BaseEstimacionMontoLicitacion` y `JustificacionMontoEstimado` se examinan mediante conteos de presencia y menciones a adjudicación o presupuesto. Estas menciones orientan la revisión y requieren interpretación del equipo. Los textos se quedan en el CSV local y no se muestran ni se exportan.

In [ ]:
if "MontoEstimadoVisible" in datos:
    visible = normalizar_codigo(datos["MontoEstimadoVisible"]).str.replace("Í", "I")
    print("Filas de oferta con monto indicado como no visible:", visible.eq("NO").sum())
    print("Proporción del total de filas:", round(visible.eq("NO").sum() / len(datos), 4) if len(datos) else 0)
    display(comparables.groupby(["MonedaLicitacion", "VisibilidadEstimado"]).agg(
        Licitaciones=("BrechaMonto", "size"),
        Coinciden=("CoincideConEstimado", "sum"),
        ProporcionCoincide=("CoincideConEstimado", "mean"),
    ))

campos_contexto = [c for c in licitaciones if c.endswith((
    "_Informada", "_MencionaAdjudicacion", "_MencionaPresupuesto",
))]
if campos_contexto:
    display(licitaciones[campos_contexto].sum().rename("Licitaciones").to_frame())
    display(comparables.groupby("CoincideConEstimado")[campos_contexto].sum())

## 7. Marcar valores extremos en CLP con IQR sobre logaritmos

Se aplica IQR al logaritmo natural de los montos positivos en CLP. El logaritmo permite trabajar con una distribución de montos asimétrica. Se reportan los límites en la escala original y cuántos casos se marcan en cada extremo. La bandera indica un **valor extremo** que requiere contexto. Se conservan todas las filas.

In [ ]:
limites = calcular_limites_iqr(datos)
datos = aplicar_bandera_atipicos(datos, limites)
if limites is not None:
    inferior = np.exp(limites["inferior_log"])
    superior = np.exp(limites["superior_log"])
    evaluadas = datos["EsExtremoLogIQR_CLP"].notna()
    montos = datos.loc[evaluadas, "MontoTotalOferta"]
    display(pd.DataFrame([{
        **limites, "Límite inferior CLP": inferior, "Límite superior CLP": superior,
        "Extremos inferiores": int(np.log(montos).lt(limites["inferior_log"]).sum()),
        "Extremos superiores": int(np.log(montos).gt(limites["superior_log"]).sum()),
    }]))
display(datos["EsExtremoLogIQR_CLP"].value_counts(dropna=False).to_frame("Ofertas"))

## 8. Validar y exportar

Se comprueban el número de filas, la cardinalidad de licitaciones, la escala ordinal, la cobertura de la brecha y de los extremos, y las columnas permitidas. El CSV de ofertas conserva el nombre de salida anterior. La tabla de licitaciones contiene la nueva brecha. Para que F3 reproduzca F2, deberá trabajar con esas dos unidades y conservar los parámetros IQR del ajuste.

In [ ]:
columnas_ofertas = [c for c in MOTIVOS if c in datos]
columnas_ofertas += ["EsDuplicadoPotencial", "TamanoProveedorOrdinal", "EsExtremoLogIQR_CLP"]
ofertas = datos[columnas_ofertas]
validar_procesado(ofertas, licitaciones, filas_originales, limites)

salida = raiz / "data/processed"
salida.mkdir(parents=True, exist_ok=True)
for tabla, nombre in [
    (ofertas, "04LicitacionesEne_procesado.csv"),
    (licitaciones, "04LicitacionesEne_brecha_por_licitacion.csv"),
]:
    destino = salida / nombre
    temporal = destino.with_suffix(".csv.tmp")
    tabla.to_csv(temporal, sep=";", index=False, encoding="utf-8-sig")
    temporal.replace(destino)
    print(f"{destino.relative_to(raiz)}: {len(tabla):,} filas × {tabla.shape[1]} columnas")